# ALL

### 4.8 RSI

While Moving Averages track the trend, they lag behind price action. to identify potential reversal points (**Overbought** or **Oversold** conditions), we introduce the **Relative Strength Index (RSI)**.

**Theoretical Basis (Mean Reversion)**:
RSI measures the speed and change of price movements. Values > 70 suggest the asset is "overheated" (potential sell), while < 30 suggest it is undervalued (potential buy).

> **Equation**: $RSI = 100 - \frac{100}{1 + RS}$, where $RS = \frac{\text{Average Gain}}{\text{Average Loss}}$

In [ ]:
# ==========================================
# [TARGET FILE]: src/features/rsi.py
# ==========================================
from src.features.rsi import add_rsi_feature

# Apply to local data
add_rsi_feature(stocks_data)

### Verification: RSI Dynamics
We inspect the RSI distribution and its relationship with the price action.

In [ ]:
# ==========================================
# [TARGET FILE]: src/features/plots.py
# ==========================================
from src.features.plots import plot_rsi_grid

# Visual Inspection
plot_rsi_grid(stocks_data)

In [ ]:
from src.features.plots import plot_feature_target_correlation

print("RSI Correlation with Target (Log_Return):")
plot_feature_target_correlation(stocks_data, feature_cols=['RSI'], title='RSI Correlation')

### 4.9 Stationary Trends (Distances)


**The Problem**: Raw prices (e.g., Apple at \$150 vs \$180) are **non-stationary**. Machine Learning models struggle to generalize on absolute values that constantly drift upwards.

**The Solution**: We convert "Trend" into a **Relative Distance**. Instead of checking if price is \$180, we check if price is **5% above its 50-day average**.

This creates a **Stationary Metric** of "Trend Extension" that works across all time periods.

In [ ]:
# ==========================================
# [TARGET FILE]: src/features/moving_average.py
# ==========================================
from src.config import FEATURE_WINDOWS
from src.features.moving_average import add_ma_distance_features

# Apply to local data
add_ma_distance_features(stocks_data, windows=FEATURE_WINDOWS)

In [ ]:
# ==========================================
# [TARGET FILE]: src/features/plots.py
# ==========================================
from src.features.plots import plot_ma_distance_grid
from src.config import FEATURE_WINDOWS

plot_win = FEATURE_WINDOWS[1] if len(FEATURE_WINDOWS) > 1 else FEATURE_WINDOWS[0]
plot_ma_distance_grid(stocks_data, window=plot_win)

In [ ]:
from src.features.plots import plot_feature_target_correlation
from src.config import FEATURE_WINDOWS

dist_cols = [f'Dist_MA{w}' for w in FEATURE_WINDOWS]
plot_feature_target_correlation(stocks_data, feature_cols=dist_cols, title='Trend Distances vs Future Return')

### 4.10 Interaction Terms (Confluence)

Markets are complex systems where factors interact. A price move accompanied by high volume is significantly more meaningful than one on low volume.

We engineer **Interaction Features** to capture this **Confluence**:

1.  **`Vol_x_Return` (Conviction)**: Weighted price move. Answers: "Did Big Money participate?"
2.  **`MACD_x_RSI` (Agreement)**: When both Trend (MACD) and Momentum (RSI) align, the probability of continuation increases.
3.  **`Trend_x_Momentum`**: Interaction between Trend Extension ($Dist_{MA}$) and Momentum ($RSI$).

> **Modeling Note (Preprocessing)**: Interaction terms often have much larger magnitudes/scales (e.g. $Volume \times Return$) than base features. 
> Since our pipeline explicitly includes a `StandardScaler` step before training, these features will be automatically normalized (Z-score), preventing magnitude bias in Linear Regression. Multicollinearity is managed via Regularization (L2 Ridge) and random feature subsampling (Random Forest).

In [ ]:
# ==========================================
# [TARGET FILE]: src/features/interactions.py
# ==========================================
from src.features.interactions import add_confluence_features
from src.features.plots import plot_interaction_correlations

# Apply to local data
add_confluence_features(stocks_data)

# Visualize Correlations
plot_interaction_correlations(stocks_data)

In [ ]:
# Detailed Interaction Correlations
from src.features.plots import plot_feature_target_correlation

interaction_features = ['Vol_x_Return', 'MACD_x_RSI', 'Trend_x_RSI']
plot_feature_target_correlation(stocks_data, feature_cols=interaction_features, title='Interaction Features Correlation')